# Extract metadata from the COP API into a DataFrame

To begin, go to this page [https://api.kb.dk/data/cop](https://api.kb.dk/data/cop). Then "select an edition", for example "Billeder" (Danish for images). If you don't want "Billeder", feel free to pick another collection. You can find more information about the different collections on [the Digital Collection.](https://digitalesamlinger.kb.dk/images/billed/2010/okt/billeder/subject2108/da/?) When you're ready, select the collection from the dropdown menu and add a search term. For example "Cirkusplakater" (Circus posters).

Press "submit", and then copy the URL from the "MODS" field.

It looks like this:

https://www.kb.dk/cop/syndication/images/billed/2010/okt/billeder/da/?itemsPerPage=30&query=Cirkusplakater&page=1&notBefore=&notAfter=&format=mods

Look carefully at the URL: it says "itemsPerPage=30". As you can read below the MODS field, the total number of items for this search is 2226. To download all the metadata for every item, you can replace "itemsPerPage=30" with "itemsPerPage=2226". You then will have a url, that will return data for all the images, that have data that matches the search term. The URL then looks like this:

https://www.kb.dk/cop/syndication/images/billed/2010/okt/billeder/da/?itemsPerPage=2226&query=Cirkusplakater&page=1&notBefore=&notAfter=&format=mods

Let's get started with the script. We begin by importing libraries.

In [1]:
# Import libraries
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
from datetime import datetime
from urllib.parse import unquote

## Get and parse MODS XML

In this cell we add the url and requests the response with requests.get, and parses the returned XML with BeautifulSoup. The list of mods elements (mods) is then used in the next cells to build the DataFrame.

In [2]:
# Send the MODS XML into a DataFrame

# URL from MODS
# NB: here I have set the itemsPerPage to 30 (itemsPerPage=30)
url = 'https://www.kb.dk/cop/syndication/images/billed/2010/okt/billeder/da/?itemsPerPage=30&query=Cirkusplakater&page=1&notBefore=&notAfter=&format=mods'

# Get data from API
response = requests.get(url)
response.raise_for_status()  # Check for errors


# Read the XML with BeautifulSoup
soup = BeautifulSoup(response.content, 'xml')

# Get all mods elements
mods = soup.find_all('mods')

## Functions to extract text

Define two functions to extract text from MODS elements:

1. get_text(elements): returns the text of the first element, or None.
2. get_all_text(elements): returns all elements' text joined with '; '.

In [ ]:
# Functions to extract text
def get_text(elements):
    """First element's text, or None."""
    if elements and len(elements) > 0:
        return elements[0].get_text(strip=True)
    return None

def get_all_text(elements):
    """All elements' text joined with '; '."""
    if elements:
        return '; '.join([e.get_text(strip=True) for e in elements if e.get_text(strip=True)])
    return None

## Parsing one MODS record

The function parse_one_mod(mod, count, all_cobject_ids) turns a single mods element into a dictionary. It reads object id and collection link, identifier tags, subject names, origin, image links (without duplicates), and a longer list of the MODS fields (see below under return).

In [4]:
def parse_one_mod(mod, count, all_cobject_ids):
    """Extract one MODS record into a dictionary with the same column names as df."""
    # Object_id and collection_link
    m = re.search(r'<\?cobject_id\s+([^?]+)\?>', str(mod))
    object_id = (m.group(1).strip() if m else (all_cobject_ids[count] if count < len(all_cobject_ids) else ''))
    collection_link = f'https://digitalesamlinger.kb.dk/{object_id}/da/'

    # Identifier tags (used in several places)
    identifier_tags = [t for t in mod.find_all('identifier') if t.name == 'identifier']

    # Person from subject (namePart under subject/name)
    subject_name_parts = []
    for subj in mod.find_all('md:subject'):
        name_el = subj.find('md:name')
        if name_el:
            np = name_el.find('md:namePart')
            if np:
                subject_name_parts.append(np.get_text(strip=True))
    subject_namePart = '; '.join(subject_name_parts) if subject_name_parts else None

    # Origin: only namePart from name with type="personal"
    name_el_personal = [n for n in mod.find_all('md:name') if n.get('type') == 'personal']
    namePart = get_all_text([n.find('md:namePart') for n in name_el_personal if n.find('md:namePart')])

    # Image links (no duplicates, preserve order)
    def links_by_label(label):
        return list(dict.fromkeys([t.get_text(strip=True) for t in identifier_tags if t.get('displayLabel') == label and t.get_text(strip=True)]))

    acc_tag = mod.find('md:identifier', {'displayLabel': 'Accession number'})
    accession_number = acc_tag.get_text(strip=True) if acc_tag else None

    return {
        'Object_id': object_id,
        'Collection_link': collection_link,
        'Titles': get_text(mod.find_all('titleInfo')),
        'Origin': namePart,
        'Ressource_type': get_text(mod.find_all('physicalDescription')),
        'Genre': get_all_text(mod.find_all('md:genre')),
        'Origin_date': get_text(mod.find_all('md:dateCreated')),
        'Topic': get_all_text(mod.find_all('md:topic')),
        'Person': subject_namePart,
        'Accession_nummer': accession_number,
        'ID': get_text(identifier_tags) if identifier_tags else None,
        'Physical_Location': get_all_text(mod.find_all('md:physicalLocation')),
        'Geographic': get_all_text(mod.find_all('md:geographic')),
        'Note': get_all_text(mod.find_all('md:note')),
        'Identifier_links_iiif': links_by_label('iiif'),
        'Identifier_links_image': links_by_label('image'),
        'Identifier_links_thumbnail': links_by_label('thumbnail'),
        'ImageHeight': get_text(mod.find_all('mix:imageHeight')),
        'ImageWidth': get_text(mod.find_all('mix:imageWidth')),
        'Relation': get_all_text(mod.find_all('md:relatedItem')),
        'AccessCondition': get_all_text(mod.find_all('md:accessCondition')),
    }

## Build the DataFrame

Get all cobject_id values from the response once, then loop over each mods element and build one record per mod. Create the DataFrame from the list of records.

In [5]:
# Get cobject_id list once, build records, create DataFrame
_all_cobject_ids = re.findall(r'<\?cobject_id\s+([^?]+)\?>', response.content.decode('utf-8'))
records = [parse_one_mod(mod, i, _all_cobject_ids) for i, mod in enumerate(mods)]
df_alt = pd.DataFrame(records)

print(f"Number of records: {len(df_alt)}")
df_alt.head()

Number of records: 30


,Object_id,Collection_link,Titles,Origin,Ressource_type,Genre,Origin_date,Topic,Person,Accession_nummer,...,Physical_Location,Geographic,Note,Identifier_links_iiif,Identifier_links_image,Identifier_links_thumbnail,ImageHeight,ImageWidth,Relation,AccessCondition
0,/images/billed/2010/okt/billeder/object1954399,https://digitalesamlinger.kb.dk//images/billed...,Cirkusplakater: Cirkus Arena,None,Plakat,None,1988-1995,None,None,None,...,Billedsamlingen. Plakatsamlingen. Cirkusplakat...,None,None,[http://kb-images.kb.dk/DAMJP2/DAM/Samlingsbil...,[http://kb-images.kb.dk/DAMJP2/DAM/Samlingsbil...,[http://kb-images.kb.dk/DAMJP2/DAM/Samlingsbil...,9967,4743,,Materialet er beskyttet af ophavsret
1,/images/billed/2010/okt/billeder/object1913515,https://digitalesamlinger.kb.dk//images/billed...,"Cirkusplakater: Cirque des Muchachos, Cirkus A...",None,Plakat,None,1970-1979,None,None,None,...,Billedsamlingen. Plakatsamlingen. Cirkusplakat...,None,None,[http://kb-images.kb.dk/DAMJP2/DAM/Samlingsbil...,[http://kb-images.kb.dk/DAMJP2/DAM/Samlingsbil...,[http://kb-images.kb.dk/DAMJP2/DAM/Samlingsbil...,10354,6770,,Materialet er beskyttet af ophavsret
2,/images/billed/2010/okt/billeder/object1915347,https://digitalesamlinger.kb.dk//images/billed...,Cirkusplakater: Circus Busch,None,Plakat,None,None,None,None,None,...,Billedsamlingen. Plakatsamlingen. Cirkusplakat...,None,None,[http://kb-images.kb.dk/DAMJP2/DAM/Samlingsbil...,[http://kb-images.kb.dk/DAMJP2/DAM/Samlingsbil...,[http://kb-images.kb.dk/DAMJP2/DAM/Samlingsbil...,13531,4858,,Materialet er beskyttet af ophavsret
3,/images/billed/2010/okt/billeder/object1915331,https://digitalesamlinger.kb.dk//images/billed...,Cirkusplakater: Circus Boswell,None,Plakat,None,None,None,None,None,...,Billedsamlingen. Plakatsamlingen. Cirkusplakat...,None,None,[http://kb-images.kb.dk/DAMJP2/DAM/Samlingsbil...,[http://kb-images.kb.dk/DAMJP2/DAM/Samlingsbil...,[http://kb-images.kb.dk/DAMJP2/DAM/Samlingsbil...,11924,7936,,Materialet er beskyttet af ophavsret
4,/images/billed/2010/okt/billeder/object1913550,https://digitalesamlinger.kb.dk//images/billed...,"Cirkusplakater: Munkefest, Munkebo",None,Plakat,None,1967,None,None,None,...,Billedsamlingen. Plakatsamlingen. Cirkusplakat...,None,None,[http://kb-images.kb.dk/DAMJP2/DAM/Samlingsbil...,[http://kb-images.kb.dk/DAMJP2/DAM/Samlingsbil...,[http://kb-images.kb.dk/DAMJP2/DAM/Samlingsbil...,9218,6493,,Materialet er beskyttet af ophavsret


## Export the DataFrame

Finally we save the DataFrame to Excel and CSV with a timestamp in the filename so each run creates a new file.

In [6]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# Extract query from URL (between query= and &page) for the filename
m = re.search(r'query=([^&]+)&page', url)
query_part = unquote(m.group(1)).strip() if m else ''
# Sanitize for filename: replace spaces and invalid chars with underscore
query_safe = re.sub(r'[\s\\/:*?"<>|]+', '_', query_part).strip('_') or 'mods'
name_base = f'mods_data_{query_safe}_{timestamp}'

excel_file = f'{name_base}.xlsx'
df_alt.to_excel(excel_file, index=False, engine='openpyxl')
print(f"DataFrame saved as Excel: {excel_file}")

# Also save as CSV
csv_file = f'{name_base}.csv'
df_alt.to_csv(csv_file, index=False, encoding='utf-8')
print(f"DataFrame saved as CSV: {csv_file}")


DataFrame saved as Excel: mods_data_Cirkusplakater_20260224_113435.xlsx
DataFrame saved as CSV: mods_data_Cirkusplakater_20260224_113435.csv
